# Лабораторная работа №2

## Тема
Обработка пропусков в данных, кодирование категориальных признаков, масштабирование данных.

## Цель
Изучить способы предварительной обработки данных для дальнейшего формирования моделей.

## Ход работы
1. Скачать подходящий датасет с пропусками и категориальными признаками.
2. Выполнить обработку пропусков.
3. Выполнить кодирование категориальных признаков.
4. Выполнить масштабирование числовых признаков.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
# Скачиваем датасет Titanic (встроенный в seaborn).
# Если загрузка через seaborn недоступна, используем прямую ссылку.

try:
    import seaborn as sns
    df = sns.load_dataset("titanic")
    source = "seaborn.load_dataset('titanic')"
except Exception:
    url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
    df = pd.read_csv(url)
    source = url

print("Источник данных:", source)
print("Размер датасета:", df.shape)
df.head()

Источник данных: seaborn.load_dataset('titanic')
Размер датасета: (891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [ ]:
# Краткий анализ структуры данных
print("Типы столбцов:")
display(df.dtypes)

print("\nКоличество пропусков по столбцам:")
display(df.isna().sum().sort_values(ascending=False))

categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print("Категориальные признаки:", categorical_cols)
print("Числовые признаки:", numeric_cols)

Типы столбцов:


survived          int64
pclass            int64
sex              object
age             float64
sibsp             int64
parch             int64
fare            float64
embarked         object
class          category
who              object
adult_male         bool
deck           category
embark_town      object
alive            object
alone              bool
dtype: object


Количество пропусков по столбцам:


deck           688
age            177
embarked         2
embark_town      2
sex              0
pclass           0
survived         0
fare             0
parch            0
sibsp            0
class            0
adult_male       0
who              0
alive            0
alone            0
dtype: int64

Категориальные признаки: ['sex', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']
Числовые признаки: ['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']


## 1) Обработка пропусков

Для числовых признаков используем заполнение **медианой**, для категориальных — **самым частым значением**.

In [ ]:
# Препроцессор для пропусков
num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

imputer_only = ColumnTransformer(
    transformers=[
        ("num", num_imputer, numeric_cols),
        ("cat", cat_imputer, categorical_cols),
    ],
    remainder="drop"
)

imputed_array = imputer_only.fit_transform(df)
imputed_columns = numeric_cols + categorical_cols
imputed_df = pd.DataFrame(imputed_array, columns=imputed_columns)

print("Пропуски после обработки:")
display(imputed_df.isna().sum().sort_values(ascending=False).head(10))

Пропуски после обработки:


survived    0
pclass      0
age         0
sibsp       0
parch       0
fare        0
sex         0
embarked    0
class       0
who         0
dtype: int64

## 2) Кодирование категориальных признаков

Используем **One-Hot Encoding** (`OneHotEncoder`) для преобразования категориальных признаков в числовой формат.

In [ ]:
# Шаг 2 (отдельно): кодирование категориальных признаков
# Сначала заполним пропуски в категориальных столбцах, затем применим One-Hot Encoding.

cat_imputer_step2 = SimpleImputer(strategy="most_frequent")
cat_filled = pd.DataFrame(
    cat_imputer_step2.fit_transform(df[categorical_cols]),
    columns=categorical_cols
)

try:
    encoder_step2 = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder_step2 = OneHotEncoder(handle_unknown="ignore", sparse=False)

cat_encoded = encoder_step2.fit_transform(cat_filled)
cat_feature_names = encoder_step2.get_feature_names_out(categorical_cols)
cat_encoded_df = pd.DataFrame(cat_encoded, columns=cat_feature_names)

print("Размер после кодирования категориальных признаков:", cat_encoded_df.shape)
cat_encoded_df.head()

## 3) Масштабирование числовых признаков

Для числовых признаков применяем стандартизацию (`StandardScaler`), чтобы привести их к сопоставимому масштабу.

In [ ]:
# Единый пайплайн: пропуски + кодирование + масштабирование
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

# Совместимость с разными версиями scikit-learn
try:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse=False)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", onehot),
    ]
)

full_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, categorical_cols),
    ],
    remainder="drop"
)

X_processed = full_preprocessor.fit_transform(df)

# Имена признаков после преобразований
feature_names = full_preprocessor.get_feature_names_out()
processed_df = pd.DataFrame(X_processed, columns=feature_names)

print("Размер после полного препроцессинга:", processed_df.shape)
processed_df.head()

Размер после полного препроцессинга: (891, 33)


,num__survived,num__pclass,num__age,num__sibsp,num__parch,num__fare,cat__sex_female,cat__sex_male,cat__embarked_C,cat__embarked_Q,...,cat__deck_E,cat__deck_F,cat__deck_G,cat__embark_town_Cherbourg,cat__embark_town_Queenstown,cat__embark_town_Southampton,cat__alive_no,cat__alive_yes,cat__alone_False,cat__alone_True
0,-0.789272,0.827377,-0.565736,0.432793,-0.473674,-0.502445,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0
1,1.266990,-1.566107,0.663861,0.432793,-0.473674,0.786845,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
2,1.266990,0.827377,-0.258337,-0.474545,-0.473674,-0.488854,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0
3,1.266990,-1.566107,0.433312,0.432793,-0.473674,0.420730,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0
4,-0.789272,0.827377,0.433312,-0.474545,-0.473674,-0.486337,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0


In [ ]:
# Проверка: среднее и стандартное отклонение у масштабированных числовых признаков
scaled_num_cols = [col for col in processed_df.columns if col.startswith("num__")]

stats = pd.DataFrame({
    "mean": processed_df[scaled_num_cols].mean(),
    "std": processed_df[scaled_num_cols].std(ddof=0),
})

print("Первые 10 числовых признаков после масштабирования:")
display(stats.head(10))

Первые 10 числовых признаков после масштабирования:


,mean,std
num__survived,-2.287732e-16,1.0
num__pclass,-2.031048e-16,1.0
num__age,3.841546e-16,1.0
num__sibsp,3.456519e-16,1.0
num__parch,6.716164e-17,1.0
num__fare,-4.373606e-17,1.0


## Вывод

В ходе лабораторной работы:
- загружен датасет `Titanic`, содержащий пропуски и категориальные признаки;
- выполнена обработка пропусков (медиана и наиболее частое значение);
- выполнено кодирование категориальных признаков методом One-Hot Encoding;
- выполнено масштабирование числовых признаков методом StandardScaler.

Данные подготовлены для последующего обучения моделей машинного обучения.